# L3 - Memory Optimization Techniques for ColPali

<p style="background-color:#fff6e4; padding:15px; border-width:3px; border-color:#f5ecda; border-style:solid; border-radius:6px"> ⏳ <b>Note <code>(Kernel Starting)</code>:</b> This notebook takes about 30 seconds to be ready to use. You may start and watch the video while you wait.</p>

<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">
<p> 💻 &nbsp; <b>Access <code>requirements.txt</code> and <code>helper.py</code> files:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Open"</em>.

<p> ⬇ &nbsp; <b>Download Notebooks:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Download as"</em> and select <em>"Notebook (.ipynb)"</em>.</p>

<p> 📒 &nbsp; For more help, please see the <em>"Appendix – Tips, Help, and Download"</em> Lesson.</p>

</div>

The following cell is not in the video and just ensures output later in this notebook will render properly.

In [ ]:
import plotly.io as pio
pio.renderers.default = "notebook"

In [ ]:
LOAD_PRECOMPUTED = True

#### Loading ColPali

In [ ]:
from colpali_engine.models import ColPaliProcessor

model_name = "vidore/colpali-v1.3"
processor = ColPaliProcessor.from_pretrained(model_name)

#### Loading Image Embedding Vectors

In [ ]:
from helper import load_sample_image_embeddings

# Load or compute image embeddings using helper function
# that only loads a sample of data
images_df = load_sample_image_embeddings(
    load_precomputed=LOAD_PRECOMPUTED,
)

# Display some random entries
sample_df = images_df.sample(n=3, random_state=42)
sample_df

#### Displaying Sample Images from the Collection and their Embedding Vectors

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

fig, axarr = plt.subplots(3, 1, figsize=(9, 16))
for i, (_, row) in enumerate(sample_df.iterrows()):
    image = Image.open(row["image_path"])
    axarr[i].imshow(image.convert("RGB"))

In [ ]:
# 检查并记录每个图像嵌入向量（Embedding）的具体维度（形状）。
# 并将获取到的维度数值存储在一个名为 original_shape 的新列中。
images_df["original_shape"] = images_df["image_embedding"].apply(
    lambda x: x.shape
)
images_df.sample(n=5)

#### Performing Vector Quantization

In [ ]:
# 向量量化（Vector Quantization）的基本原理。
# 量化的目的是为了压缩数据，减少内存占用，并加快向量搜索的速度。
import numpy as np

# Perform bucketing
# 标量量化
min_value, max_value = -0.8, 1.2
value_space = np.linspace(min_value, max_value, 256)

# Find which bucket index the target value belongs to
# 在实际应用中，我们不再存储 0.2 这个占用空间大的浮点数，而是只存储 128 这个索引值（只需 1 个字节）。
target_value = 0.2
bucket_index = np.argmin(np.abs(value_space - target_value))
bucket_index

In [ ]:
# 二进制量化
simple_vector = 2 * np.random.random(16) - 1.0
binary_vector = (simple_vector > 0).astype(int)
print(simple_vector, "->", binary_vector)

#### Displaying a Sample Image's Tokens and Embedding Vectors

In [ ]:
# 多模态 AI 模型（如 ColPali 或 PaliGemma）是如何在底层将一张图像转化成计算机可处理的“Token”（令牌）序列的
image = Image.open(images_df["image_path"][0])
batch_images = processor.process_images([image])
processor.decode(batch_images.input_ids[0])

In [ ]:
# 从混合了多种信息的向量序列中，精准提取出属于“图像视觉特征”的那部分嵌入向量（Embeddings）

# get_image_mask 函数会生成一个“掩码”（Mask），它是一个布尔数组（True/False）。
# 在这个数组中，只有对应**真实图像块（Patches）**的位置是 True，其他位置（如补白或特殊字符）是 False。
image_mask = processor.get_image_mask(batch_images)[0]
images_df["image_embedding"][0][image_mask]

In [ ]:
# 将原本“排成一队”的特征向量，重新拼回成一张“特征地图”（Grid）。
# 在之前的步骤中，模型把图像切成了很多小方块（Patches），但处理时是将它们当作一个一维的长序列（就像把书拆开，所有字排成一长排）。

# 说明模型将图像横向切了 32 份，纵向也切了 32 份，总共 32×32=1024 个小方块。
patch_size = 32  # the default Colpali patch size
# 128 是每个小块对应的特征向量长度。
model_dim = 128  # the default Colpali embedding size

# 为什么要这么做？（还原“地理位置”）当你把向量重构成 32×32 的网格后，数据就具备了空间对应关系：
# 网格中 [0, 0] 位置的向量，对应的就是原始 PDF 图像的左上角。
# 网格中 [31, 31] 位置的向量，对应的就是图像的右下角。
# 如果不做这一步，模型只知道图像里有什么，但不知道东西在哪。
# 做了这一步，程序就能精准定位到“‘Andrew Ng’这几个字出现在幻灯片的右下角”或者“图表在幻灯片的中间”。

def embeddings_grid(image_embeddings: np.ndarray):
    """
    Reshape the image embeddings so we have a grid of patches
    and their corresponding embeddings.
    """
    return image_embeddings.reshape((patch_size, patch_size, model_dim))

In [ ]:
# 先取第一张图的原始数据
# 然后利用上一步提到的掩码，把 1031 个 Token 中没用的填充字符删掉，刚好剩下 1024 个视觉特征 Token
# 调用 embeddings_grid 函数，把这 1024 个点拼成 32×32 的方阵。
grid = embeddings_grid(images_df["image_embedding"][0][image_mask])
grid

#### Performing Row and Column Pooling

In [ ]:
# 平均池化（Mean Pooling）操作，专门用于对图像特征进行降维和布局特征提取。

# 行平均池化
# 它沿着水平方向计算平均值。
# 也就是说，它把每一行中的 32 个小方块的向量加起来求平均。
# 物理意义：这代表了图像的 “水平切片特征”。
# 例如：如果 PDF 的第一行是标题，那么 row_mean_pooling 的第一个向量就会包含明显的“标题文字”特征。
# 这对于识别文档的行结构（如段落分布、标题位置）非常有用。
def row_mean_pooling(grid_embeddings: np.ndarray) -> np.ndarray:
    return grid_embeddings.mean(axis=1)

# 列平均池化
# 它沿着垂直方向计算平均值。
# 把每一列中的 32 个小方块的向量求平均。
# 物理意义：这代表了图像的 “垂直切片特征”。
# 例如：如果 PDF 是分两栏排版的，那么中间那一列的平均向量特征会非常弱（因为是空白），而左右两栏的特征会很强。
# 这对于识别文档的分栏布局、侧边栏位置非常有用。
def column_mean_pooling(grid_embeddings: np.ndarray) -> np.ndarray:
    return grid_embeddings.mean(axis=0)

# 总结：为什么要这么做？
# 这种处理方式常见于**文档检索（Document AI）**领域，主要有以下三个目的：
# 特征压缩：把 1024 个向量压缩成 32个，计算量直接减少了 32 倍，大大加快了后续搜索或比对的速度。
# 捕获布局信息：通过行/列池化，AI 可以更轻松地识别出文档的“骨架”。比如通过查看列特征，AI 能立刻判断出这是一个“单栏”还是“双栏”的页面。
# 对齐与匹配：在处理像 ColPali 这样的模型时，这种池化后的特征可以用来进行粗筛，快速锁定可能包含目标信息的页面区域。

In [ ]:
row_mean_pooling(grid)

In [ ]:
column_mean_pooling(grid)

#### Performing Hierarchical Pooling

In [ ]:
# 对图像嵌入向量进行“分层池化”（Hierarchical Pooling），以实现数据压缩。


from colpali_engine.compression.token_pooling import (
    # colpali_engine 专门提供的工具。它的作用是将相邻或相似的 Token 融合在一起。
    HierarchicalTokenPooler,
)

import numpy as np
import torch

pooler = HierarchicalTokenPooler()


def hierarchical_token_pooling(
    arr: np.ndarray, pool_factor: int = 2      # 压缩因子。设置为 2 意味着它尝试将 Token 的数量减少一半左右。    
) -> np.ndarray:
    """
    Apply hierarchical clustering to a single document embedding.
    """
    # Convert the array to 3D torch tensor
    # 将 NumPy 数组转成了 PyTorch 张量（Tensor），因为池化算法是在深度学习框架上跑的。
    arr_tensor = torch.from_numpy(arr[np.newaxis, :, :])

    # Apply hierarchical pooling
    # 算法会扫描那 1031 个向量，每两个（或根据算法逻辑）合并成一个更具有代表性的新向量。
    pooled = pooler.pool_embeddings(arr_tensor, pool_factor=pool_factor)
    return pooled.cpu().detach().numpy()[0]

In [ ]:
# Demonstrate on a single example
example_embedding = images_df["image_embedding"][0]
print(f"Original shape: {example_embedding.shape}")

# Apply hierarchical token pooling with pool_factor=2
pooled_embedding = hierarchical_token_pooling(
    example_embedding, pool_factor=2
)
print(f"Pooled shape: {pooled_embedding.shape}")
# 原始形状 (Original shape)：(1031, 128)。即 1031 个 128 维的向量。
# 池化后形状 (Pooled shape)：(515, 128)。
# 计算：1031 ÷ 2 = 515.5。
# 代码将其压缩到了 515 个向量。
# 结果：数据量直接减少了 50%。

# 通常，开发者会在存入 Qdrant 之前的最后一刻，才使用 Mask 彻底清除那些已经随之缩小的“残余填充位”。

#### Creating Qdrant Collection and Adding Vectors

In [ ]:
# 连接到运行在本地（localhost:6333）的 Qdrant 数据库。
# 如果集合已存在，先删除旧的，确保实验环境是干净的。
from qdrant_client import QdrantClient, models

collection_name = "colpali-optimizations"

# Connect to Qdrant
client = QdrantClient("http://localhost:6333", timeout=120)

# Delete collection if it exists
if client.collection_exists(collection_name):
    client.delete_collection(collection_name)

In [ ]:
# 在一个集合中，代码定义了 7 个“命名向量”（Named Vectors），每个向量对应一种优化策略。
# 它们都使用了 ColPali 模型核心的 Multi-Vector（多向量） 配置和 MAX_SIM（最大相似度） 比较算法。
client.create_collection(
    collection_name,
    vectors_config={
        "original": models.VectorParams(
            size=128,
            distance=models.Distance.DOT,
            multivector_config=models.MultiVectorConfig(
                comparator=models.MultiVectorComparator.MAX_SIM,
            ),
            hnsw_config=models.HnswConfigDiff(m=0),
            on_disk=True,
        ),
        "scalar_quantized": models.VectorParams(
            size=128,
            distance=models.Distance.DOT,
            multivector_config=models.MultiVectorConfig(
                comparator=models.MultiVectorComparator.MAX_SIM,
            ),
            hnsw_config=models.HnswConfigDiff(m=0),
            # Enable Scalar Quantization for a single named vector
            # and not the entire collection
            quantization_config=models.ScalarQuantization(
                scalar=models.ScalarQuantizationConfig(
                    type=models.ScalarType.INT8,
                ),
            ),
            on_disk=True,
        ),
        "binary_quantized": models.VectorParams(
            size=128,
            distance=models.Distance.DOT,
            multivector_config=models.MultiVectorConfig(
                comparator=models.MultiVectorComparator.MAX_SIM,
            ),
            hnsw_config=models.HnswConfigDiff(m=0),
            # Enable Binary Quantization for a single named vector
            # and not the entire collection
            quantization_config=models.BinaryQuantization(
                binary=models.BinaryQuantizationConfig(
                    always_ram=True,
                ),
            ),
            on_disk=True,
        ),
        "hierarchical_2x": models.VectorParams(
            size=128,
            distance=models.Distance.DOT,
            multivector_config=models.MultiVectorConfig(
                comparator=models.MultiVectorComparator.MAX_SIM,
            ),
            hnsw_config=models.HnswConfigDiff(m=0),
            on_disk=True,
        ),
        "hierarchical_4x": models.VectorParams(
            size=128,
            distance=models.Distance.DOT,
            multivector_config=models.MultiVectorConfig(
                comparator=models.MultiVectorComparator.MAX_SIM,
            ),
            hnsw_config=models.HnswConfigDiff(m=0),
            on_disk=True,
        ),
        "row_pooled": models.VectorParams(
            size=128,
            distance=models.Distance.DOT,
            multivector_config=models.MultiVectorConfig(
                comparator=models.MultiVectorComparator.MAX_SIM,
            ),
            hnsw_config=models.HnswConfigDiff(m=0),
            on_disk=True,
        ),
        "column_pooled": models.VectorParams(
            size=128,
            distance=models.Distance.DOT,
            multivector_config=models.MultiVectorConfig(
                comparator=models.MultiVectorComparator.MAX_SIM,
            ),
            hnsw_config=models.HnswConfigDiff(m=0),
            on_disk=True,
        ),
    },
)

<p style="background-color:#fff6e4; padding:15px; border-width:3px; border-color:#f5ecda; border-style:solid; border-radius:6px"> ⏳ Expect the following cell will take a few minutes to finish</p>

In [ ]:
# 整个流程的**“数据入库”**阶段。
# 它的作用是将之前处理好的所有图像路径及其对应的 7 种向量方案，正式上传（Upsert）到 Qdrant 向量数据库中。
from helper import yield_optimized_embeddings
from tqdm import tqdm

allowed_docs = (
    "AI4E_W1", "AI4E_W2", "DLS_C4W4", "GenAI4E_W1",
    "MLS_C2_W1", "MLS_C3_W2", "MLS_C2_W3", "DLS_C3_W1",
    "DLS_C1_W1", "MLS_C3_W3", "MLS_C1_W1",
)

# Use the generator to load optimized embeddings efficiently
# Set load_precomputed=False the first time to compute and save optimizations
# Set load_precomputed=True to load from saved files (much faster!)
for i, (image_path, vectors) in enumerate(
    tqdm(
        # 它会逐一读取图像，并根据之前的逻辑计算出那 7 种优化后的向量。
        yield_optimized_embeddings(load_precomputed=LOAD_PRECOMPUTED, 
                                   allowed_docs=allowed_docs),
        desc="Upserting embeddings"
    )
):
    # Insert each embedding with all optimization variants
    client.upsert(
        collection_name,
        points=[
            models.PointStruct(
                # 给每张图一个唯一的数字编号。
                id=i,
                # 一次性存入了 7 个版本的向量。
                vector={
                    "original": vectors["original"],
                    # Scalar and Binary Quantization are handled internally
                    # by Qdrant engine, so we send original vector
                    "scalar_quantized": vectors["original"],
                    "binary_quantized": vectors["original"],
                    "hierarchical_2x": vectors["hierarchical_2x"],
                    "hierarchical_4x": vectors["hierarchical_4x"],
                    "row_pooled": vectors["row_pooled"],
                    "column_pooled": vectors["column_pooled"],
                },
                # 存储非向量数据，这里存的是 image_path。这样当你搜索到某个向量时，数据库能告诉你这张图在磁盘上的具体位置。
                payload={
                    "image_path": image_path,
                },
            )
        ],
    )

# 我们在最下方看到的红色进度条就是由它生成的，显示总共处理了 708 张图像，耗时约 2 分 25 秒。    

#### Loading Sample Query Vectors

In [ ]:
from helper import load_or_compute_query_embeddings

# Load or compute query embeddings using helper function
queries_df = load_or_compute_query_embeddings(
    load_precomputed=LOAD_PRECOMPUTED,
)

# Extract queries and query embeddings for later use
queries = queries_df["query"].tolist()
query_embeddings = queries_df["query_embedding"].tolist()

# Display the queries
queries_df

In [ ]:
# Define all optimization methods to compare
vector_names = [
    "original",  # Always first (baseline)
    "scalar_quantized",
    "binary_quantized",
    "row_pooled",
    "column_pooled",
    "hierarchical_2x",
    "hierarchical_4x",
]

#### Comparing Different Optimization Methods

In [ ]:
from helper import compare_optimization_methods

# Query 1: Coffee mug
fig = compare_optimization_methods(
    query=queries[0],
    query_embedding=query_embeddings[0],
    client=client,
    collection_name=collection_name,
    vector_names=vector_names,
    limit=3,
)
fig.show()

In [ ]:
# Query 2: Size vs performance tradeoff
fig = compare_optimization_methods(
    query=queries[1],
    query_embedding=query_embeddings[1],
    client=client,
    collection_name=collection_name,
    vector_names=vector_names,
    limit=3,
)
fig.show()

In [ ]:
# Query 3: One learning algorithm
fig = compare_optimization_methods(
    query=queries[2],
    query_embedding=query_embeddings[2],
    client=client,
    collection_name=collection_name,
    vector_names=vector_names,
    limit=3,
)
fig.show()